In [4]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import StratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix

)

from scipy.stats import friedmanchisquare
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

In [5]:
# LOAD DATASET AND DEFINE DATASET
df = pd.read_csv("INCART 2-lead Arrhythmia Database.csv")
print(df.shape)
df.head()

(175729, 34)


,record,type,0_pre-RR,0_post-RR,0_pPeak,0_tPeak,0_rPeak,0_sPeak,0_qPeak,0_qrs_interval,...,1_qPeak,1_qrs_interval,1_pq_interval,1_qt_interval,1_st_interval,1_qrs_morph0,1_qrs_morph1,1_qrs_morph2,1_qrs_morph3,1_qrs_morph4
0,I01,N,163,165,0.069610,-0.083281,0.614133,-0.392761,0.047159,15,...,-0.023370,14,3,23,6,-0.023370,-0.011650,0.082608,0.101373,-0.183387
1,I01,N,165,166,-0.097030,0.597254,-0.078704,-0.078704,-0.137781,3,...,0.081637,15,5,27,7,0.081637,0.102992,0.191225,0.217544,-0.068248
2,I01,N,166,102,0.109399,0.680528,-0.010649,-0.010649,-0.720620,6,...,-0.148539,33,13,52,6,-0.148539,-0.060620,0.081080,0.204400,0.335172
3,I01,VEB,102,231,0.176376,0.256431,-0.101098,-0.707525,-0.101098,4,...,0.046898,21,9,34,4,0.046898,0.083728,0.279512,0.526785,0.450969
4,I01,N,231,165,0.585577,0.607461,-0.083499,-0.083499,-0.167858,3,...,-0.112552,32,5,43,6,-0.112552,0.012989,0.091491,0.134004,0.265232


In [6]:
#DATA PREPROCESSING
# Remove duplicate rows
df = df.drop_duplicates()

# Missing values
df.isnull().sum()

# Remove identifier column
df.drop(columns=['record'], inplace=True)

# Separate X and y
X = df.drop("type", axis=1)
y = df["type"]

# Encode target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [7]:
models={
'Logistic Regression':
LogisticRegression(
    C=0.1,
    max_iter=300,
    random_state=42
),

'KNN':
KNeighborsClassifier(
    n_neighbors=3
),

'Decision Tree':
DecisionTreeClassifier(
    max_depth=10,
    random_state=42
),

'Random Forest':
RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
),

'SVM':
CalibratedClassifierCV(
LinearSVC(
    C=1,
    dual=False,
    max_iter=2000,
    random_state=42
),
cv=3
),

'MLP':

MLPClassifier(
    hidden_layer_sizes=(50,),
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42
),

'XGBoost':
XGBClassifier(
    n_estimators=50,
    max_depth=5,
    learning_rate=0.1,
    tree_method='hist',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

}

In [8]:
metric_names=[
'Accuracy',
'Precision',
'Recall',
'F1',
'ROC',
'Kappa',
'MCC',
'GMean'
]
results={
metric:{
model:[]
for model in models
}
for metric in metric_names
}

In [9]:
cv=StratifiedKFold(
n_splits=2,
shuffle=True,
random_state=42
)


smote = SMOTE(
    random_state=42,
    k_neighbors=1
)

In [10]:
for train_idx,test_idx in cv.split(X,y):


    X_train=X.iloc[

        train_idx

    ]

    X_test=X.iloc[

        test_idx

    ]

    y_train=y[

        train_idx

    ]

    y_test=y[

        test_idx

    ]


    X_train_smote,y_train_smote=smote.fit_resample(X_train,y_train)


    for name,model in models.items():
        print("\nTraining Model:", name)
        model.fit(

            X_train_smote,

            y_train_smote

        )


        pred=model.predict(

            X_test

        )


        prob=model.predict_proba(

            X_test

        )


        acc=accuracy_score(

            y_test,

            pred

        )


        prec=precision_score(

            y_test,

            pred,

            average='weighted',

            zero_division=0

        )


        rec=recall_score(

            y_test,

            pred,

            average='weighted'

        )


        f1=f1_score(

            y_test,

            pred,

            average='weighted'

        )


        roc=roc_auc_score(

            y_test,

            prob,

            multi_class='ovr',

            average='weighted'

        )


        kap=cohen_kappa_score(

            y_test,

            pred

        )


        mcc=matthews_corrcoef(

            y_test,

            pred

        )


        cm=confusion_matrix(

            y_test,

            pred

        )


        sens=np.diag(

            cm

        )/cm.sum(

            axis=1

        )


        sens=np.nan_to_num(

            sens

        )


        gmean=np.prod(

            sens

        )**(

            1/len(sens)

        )


        results['Accuracy'][name].append(acc)

        results['Precision'][name].append(prec)

        results['Recall'][name].append(rec)

        results['F1'][name].append(f1)

        results['ROC'][name].append(roc)

        results['Kappa'][name].append(kap)

        results['MCC'][name].append(mcc)

        results['GMean'][name].append(gmean)




Training Model: Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Training Model: KNN

Training Model: Decision Tree

Training Model: Random Forest

Training Model: SVM

Training Model: MLP

Training Model: XGBoost

Training Model: Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Training Model: KNN

Training Model: Decision Tree

Training Model: Random Forest

Training Model: SVM

Training Model: MLP

Training Model: XGBoost


In [12]:
friedman=[]


for metric in metric_names:


    stat,p=friedmanchisquare(*[results[metric][model]for model in models])
    row={
'Metric':metric,
'Statistic':stat,
'P_Value':p,
'Significant':p<0.05
    }

    for model in models:
        row[
model+'_Mean'
]=round(
np.mean(
results[metric][model]
),
4
)
    friedman.append(
row
)
friedman_df=\
pd.DataFrame(
friedman
)


print()
print(friedman_df)

friedman_df.to_excel(
'FriedmanResults.xlsx',
index=False
)



      Metric  Statistic   P_Value  Significant  Logistic Regression_Mean  \
0   Accuracy  11.785714  0.066924        False                    0.8394   
1  Precision  11.785714  0.066924        False                    0.9718   
2     Recall  11.785714  0.066924        False                    0.8394   
3         F1  12.000000  0.061969        False                    0.8983   
4        ROC  12.000000  0.061969        False                    0.9808   
5      Kappa  11.785714  0.066924        False                    0.5332   
6        MCC  11.785714  0.066924        False                    0.5907   
7      GMean   6.000000  0.423190        False                    0.3385   

   KNN_Mean  Decision Tree_Mean  Random Forest_Mean  SVM_Mean  MLP_Mean  \
0    0.9863              0.9615              0.9863    0.9737    0.9889   
1    0.9877              0.9816              0.9918    0.9822    0.9923   
2    0.9863              0.9615              0.9863    0.9737    0.9889   
3    0.9869   